# Time series analysis

In [ ]:
%pip install git+https://github.com/bobiac/bobiac-tools.git
%pip install matplotlib
%pip install numpy
%pip install scikit-image
%pip install scipy
%pip install tifffile
%pip install imagecodecs
%pip install pandas
%pip install seaborn

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import skimage
import tifffile
from bobiac_tools import overlay_labels

## Overview

In this notebook, we analyse time resolved data.

We have data of cells where the location of our gene of interes oscillates between the nucleus and the cytoplasm. Our question is **What is the frequency of the oscillations?**

We have a total of 16 time points with a temporal spacing of 30 min.
The background of the images was measured to be 4700.

In [ ]:
image_folder = Path("https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/quant/03_time_series/images/")
image_paths = sorted(image_folder.glob("*.tif"))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, path in zip(axes.flat, image_paths[:8]):
    image = tifffile.imread(path)
    tp = path.stem.split("_t")[1].split("_")[0]
    ax.imshow(image, cmap="gray")
    ax.set_title(f"t={tp}", fontsize=20)
    ax.axis("off")

plt.tight_layout()
plt.show()

These are the first eight time points.

In [ ]:
mask_cells = tifffile.imread(
    "https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/quant/03_time_series/masks/F01_202_cells.TIF"
)
mask_nuclei = tifffile.imread(
    "https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/quant/03_time_series/masks/F01_202_nuclei.TIF"
)

overlay_labels(label_mask=[mask_cells, mask_nuclei])

This are the two label masks for the nuclei and the cells.

### ✍️ Exercise: Perform this analysis

1. For the first time point (image) use `regionprops_table` to measure the expression level of each nucleus. Follow the steps of the previous notebooks.
2. Perform proper quality control and check that the measurements do not contain artefacts.
3. Set up batch processing to analyse all 16 images. Remember to keep track of where the data came from. Add a column `time` that contains the time at which the image was taken (e.g. 0, 0.5, 1, 1.5) (the unit is hours).
4. Use `sns.lineplot` to visualise the oscillations of individual cell as well as the average oscillations, and estimate the frequency.
5. BONUS: Instead of nuclear intensity, calculate the ratio of nuclear/cytoplasmic intensity.

## Solution

### 1. Measure nuclear intensity
First, we measure the intesities of all the nuclei in the image.

Load the first time point.

In [ ]:
image_path = image_paths[0]

image_t0 = tifffile.imread(image_path)
overlay_labels(image=image_t0)

Remove boundary objects.

In [ ]:
mask_nuclei = skimage.segmentation.clear_border(mask_nuclei)
overlay_labels(image=image_t0, label_mask=mask_nuclei)

Measure intesities with `regionprops_table` ans save measurements as `DataFrame`.

In [ ]:
properties = ["area", "intensity_mean", "label"]
props = skimage.measure.regionprops_table(
    mask_nuclei, intensity_image=image_t0, properties=properties
)

df_nuc = pd.DataFrame(props)
print(df_nuc)

Subtract background intensity from measurements. Background was measured to 4700.

In [ ]:
background_intensity = 4700
df_nuc["intensity_bg_cor"] = df_nuc["intensity_mean"] - background_intensity

Add `time` column.

In [ ]:
time_point = image_path.stem.split("_")[2]
time = float(time_point[1:]) * 0.5
print(time)

In [ ]:
df_nuc["time"] = time

### 2. Quality control

In [ ]:
sns.histplot(
    data=df_nuc,
    x="intensity_bg_cor",
    bins=20,
)

Seems unsuspicious.

In [ ]:
overlay_labels(
    image=image_t0,
    label_mask=mask_nuclei,
    df=df_nuc,
    id_col="label",
    measurement_col="intensity_bg_cor",
)

Looks fine.

### Batch processing

In [ ]:
list_df = []

for image_path in image_paths:
    image = tifffile.imread(image_path)
    props = skimage.measure.regionprops_table(
        mask_nuclei, intensity_image=image, properties=properties
    )
    sdf = pd.DataFrame(props)
    sdf["intensity_bg_cor"] = sdf["intensity_mean"] - background_intensity
    time_point = image_path.stem.split("_")[2]
    time = float(time_point[1:]) * 0.5
    sdf["time"] = time
    list_df.append(sdf)

df = pd.concat(list_df)
print(df)

### Plotting

In [ ]:
df.groupby(["label", "time"])["intensity_bg_cor"].mean().reset_index()

In [ ]:
sns.lineplot(data=df, x="time", y="intensity_bg_cor", errorbar="sd")

In [ ]:
sns.lineplot(data=df, x="time", y="intensity_bg_cor", hue="label")